# KNN Training

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)
import os
import joblib
import json

## Loading Data

In [2]:
X_train=pd.read_csv("X_train.csv")
Y_train=pd.read_csv("Y_train.csv")
X_test=pd.read_csv("X_test.csv")
Y_test=pd.read_csv("Y_test.csv")

## Training

In [3]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [4]:
knn= KNeighborsClassifier(
    n_neighbors=5, 
    n_jobs=-1, 
    metric='minkowski', 
    p=2, 
    leaf_size=30
)

In [5]:
print("Training KNN...")
knn.fit(X_train_scaled, Y_train)
print("Training completed.")

Training KNN...
Training completed.


C:\Users\ali\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\neighbors\_classification.py:243: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)


In [6]:
Y_pred = knn.predict(X_test_scaled)
Y_prob = knn.predict_proba(X_test_scaled)[:, 1]

## Model Evaluation

In [7]:
print("\nClassification Report:")
print(classification_report(
    Y_test,
    Y_pred,
    target_names=["Benign", "Non-Benign"]
))

print("Accuracy:", accuracy_score(Y_test, Y_pred))
print("Precision:", precision_score(Y_test, Y_pred))
print("Recall:", recall_score(Y_test, Y_pred))
print("F1 Score:", f1_score(Y_test, Y_pred))
print("ROC-AUC:", roc_auc_score(Y_test, Y_prob))
print("PR-AUC:", average_precision_score(Y_test, Y_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(Y_test, Y_pred))


Classification Report:
              precision    recall  f1-score   support

      Benign       1.00      1.00      1.00     84976
  Non-Benign       0.44      0.76      0.56       142

    accuracy                           1.00     85118
   macro avg       0.72      0.88      0.78     85118
weighted avg       1.00      1.00      1.00     85118

Accuracy: 0.9979675274325055
Precision: 0.43724696356275305
Recall: 0.7605633802816901
F1 Score: 0.5552699228791774
ROC-AUC: 0.8867828215290612
PR-AUC: 0.557638938779394

Confusion Matrix:
[[84837   139]
 [   34   108]]


## Saving Model

In [9]:
os.makedirs("models", exist_ok=True)

joblib.dump(
    knn,
    "models/KNN_model.pkl"
)

['models/KNN_model.pkl']

## Saving Metrics

In [10]:
os.makedirs("KNN", exist_ok=True)
metrics = {
    "accuracy": accuracy_score(Y_test, Y_pred),
    "precision": precision_score(Y_test, Y_pred),
    "recall": recall_score(Y_test, Y_pred),
    "f1_score": f1_score(Y_test, Y_pred),
    "roc_auc": roc_auc_score(Y_test, Y_prob),
    "pr_auc": average_precision_score(Y_test, Y_prob)
}

with open("KNN/metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

print("Metrics saved.")

Metrics saved.


In [11]:
cm = confusion_matrix(Y_test, Y_pred)

cm_df = pd.DataFrame(
    cm,
    index=["Actual_Positive", "Actual_Negative"],
    columns=["Predicted_Positive", "Predicted_Negative"]
)

cm_df.to_csv(
    "KNN/confusion_matrix.csv"
)

print("Confusion matrix saved.")

Confusion matrix saved.


In [12]:
report = classification_report(
    Y_test,
    Y_pred,
    target_names=["Benign", "Non-Benign"]
)

with open("KNN/classification_report.txt", "w") as f:
    f.write(report)

print("Classification report saved.")

Classification report saved.


## Saving Predictions

In [13]:
predictions = pd.DataFrame({
    "actual": Y_test.values.ravel(),
    "prediction": Y_pred,
    "probability_non_benign": Y_prob
})

predictions.to_csv(
    "KNN/predictions.csv",
    index=False
)

## Saving Scaler

In [14]:
joblib.dump(scaler,"models/KNNscaler.pkl")

['models/KNNscaler.pkl']